## Descarga y exploración inicial de los datos - Enfermedad del corazón (Heart Disease)

### By:
Ronaldo Duran (JRDT)

### Date:
2026-08-15

### Description:

Notebook de obtención y entendimiento de los datos RAW del proyecto de clasificación
de enfermedad cardíaca. Se carga el dataset `corazon.csv`, se documenta su fuente y su
diccionario de variables, se hace un primer diagnóstico de calidad de los datos y se
responde el checklist de entendimiento del problema definido en el
[issue #7 - Descarga de datos](https://github.com/ronaldo-duran/Hearth-project/issues/7).

Los datos se guardan en un formato local eficiente (parquet) para las siguientes
iteraciones del proyecto.

## 📚 Import libraries

In [1]:
# base libraries for data science
from pathlib import Path

import pandas as pd

## 💾 Load data

In [2]:
# Raíz del proyecto: se busca hacia arriba la carpeta data/01_raw
current_dir = Path.cwd().resolve()
project_root = next(
    p for p in [current_dir, *current_dir.parents] if (p / "data" / "01_raw").exists()
)

DATA_DIR = project_root / "data" / "01_raw"
file_path = DATA_DIR / "corazon.csv"

corazon_df = pd.read_csv(file_path)
corazon_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   age         3000 non-null   str    
 1   sex         2969 non-null   str    
 2   chest_pain  2947 non-null   str    
 3   rest_bp     2949 non-null   str    
 4   chol        2945 non-null   str    
 5   fbs         2933 non-null   float64
 6   rest_ecg    2837 non-null   str    
 7   max_hr      2859 non-null   str    
 8   exang       2879 non-null   str    
 9   old_peak    2880 non-null   str    
 10  slope       2879 non-null   str    
 11  ca          2868 non-null   str    
 12  thal        2904 non-null   str    
 13  disease     2924 non-null   str    
dtypes: float64(1), str(13)
memory usage: 506.9 KB


## 📊 Data info

### Fuente de los datos

- **Dataset**: versión **modificada** del dataset *Heart Disease UCI* de Kaggle
  ([ronitf/heart-disease-uci](https://www.kaggle.com/ronitf/heart-disease-uci)), que a su
  vez proviene del repositorio UCI Machine Learning Repository (Heart Disease, hospital
  Cleveland, ~303 pacientes). La versión modificada del curso tiene **3030 filas** y
  añade problemas de calidad deliberados.
- **Nota importante** (según `data/01_raw/datos_corazon_Info.txt`): **no usar el dataset
  original**, la copia local `data/01_raw/corazon.csv` es la fuente única de verdad.
- **Cada fila** representa los exámenes de un paciente junto con la variable objetivo
  `disease` (1 = tiene enfermedad cardíaca, 0 = no la tiene).
- El objetivo del sistema es determinar **automáticamente** si un paciente tiene
  problemas del corazón a partir de los resultados de sus exámenes.

In [3]:
corazon_df.head()

,age,sex,chest_pain,rest_bp,chol,fbs,rest_ecg,max_hr,exang,old_peak,slope,ca,thal,disease
0,63,Male,typical,145,233,1.0,left ventricular hypertrophy,150,0,2.3,3,0.0,fixed,0
1,67,Male,asymptomatic,160,286,0.0,left ventricular hypertrophy,108,1,1.5,2,3.0,normal,1
2,67,Male,asymptomatic,120,229,0.0,left ventricular hypertrophy,129,1,2.6,2,2.0,reversable,1
3,37,Male,nonanginal,130,250,0.0,normal,187,0,3.5,3,0.0,normal,0
4,41,Female,nontypical,130,204,0.0,left ventricular hypertrophy,172,0,1.4,1,0.0,normal,0


In [4]:
print(f"Dimensiones: {corazon_df.shape[0]} filas x {corazon_df.shape[1]} columnas")
corazon_df.dtypes.to_frame("dtype")

Dimensiones: 3030 filas x 14 columnas


,dtype
age,str
sex,str
chest_pain,str
rest_bp,str
chol,str
fbs,float64
rest_ecg,str
max_hr,str
exang,str
old_peak,str



### Diccionario de variables

| Variable | Tipo | Descripción |
| --- | --- | --- |
| `age` | numérica | Edad en años |
| `sex` | categórica | Male / Female |
| `chest_pain` | categórica | Tipo de dolor torácico: typical, asymptomatic, nonanginal, nontypical |
| `rest_bp` | numérica | Presión arterial en reposo (mm Hg al ingreso al hospital) |
| `chol` | numérica | Colesterol sérico en mg/dl |
| `fbs` | binaria | Glucosa en ayunas > 120 mg/dl (1 = true, 0 = false) |
| `rest_ecg` | categórica | Resultado del electrocardiograma en reposo |
| `max_hr` | numérica | Frecuencia cardíaca máxima alcanzada |
| `exang` | binaria | Angina inducida por el ejercicio (1 = yes, 0 = no) |
| `old_peak` | numérica | Depresión del segmento ST inducida por el ejercicio respecto al reposo |
| `slope` | ordinal | Pendiente del segmento ST en el pico del ejercicio |
| `ca` | numérica (0-3) | Número de vasos principales coloreados por fluoroscopia |
| `thal` | categórica | Gammagrafía con talio: normal, fixed defect, reversable defect |
| `disease` | binaria (**target**) | 1 = enfermedad cardíaca, 0 = sin enfermedad |

### Diagnóstico de calidad de los datos

El dataset está modificado a propósito, por lo que antes de modelar hay que
catalogar sus problemas: valores nulos, filas duplicadas y valores corruptos.

In [5]:
# Valores nulos por columna
print(f"Filas con al menos un nulo: {corazon_df.isna().any(axis=1).sum()}")
print(f"Filas completamente vacias: {corazon_df.isna().all(axis=1).sum()}")
corazon_df.isna().sum().to_frame("nulos")

Filas con al menos un nulo: 318
Filas completamente vacias: 15


,nulos
age,30
sex,61
chest_pain,83
rest_bp,81
chol,85
fbs,97
rest_ecg,193
max_hr,171
exang,151
old_peak,150


In [6]:
# Filas duplicadas
n_duplicates = corazon_df.duplicated().sum()
print(
    f"Filas duplicadas exactas: {n_duplicates} de {len(corazon_df)} ({n_duplicates / len(corazon_df):.1%})"
)
print(f"Filas únicas: {len(corazon_df.drop_duplicates())}")

# Exámenes idénticos con etiquetas contradictorias
features = [col for col in corazon_df.columns if col != "disease"]
label_conflicts = (corazon_df.dropna().groupby(features)["disease"].nunique() > 1).sum()
print(f"Grupos de exámenes idénticos con etiquetas distintas (contradictorias): {label_conflicts}")

Filas duplicadas exactas: 2462 de 3030 (81.3%)
Filas únicas: 568
Grupos de exámenes idénticos con etiquetas distintas (contradictorias): 5


In [7]:
# Valores no numéricos en columnas que deberían ser numéricas
numeric_columns = [
    "age",
    "rest_bp",
    "chol",
    "fbs",
    "max_hr",
    "exang",
    "old_peak",
    "slope",
    "ca",
    "disease",
]
for col in numeric_columns:
    coerced = pd.to_numeric(corazon_df[col], errors="coerce")
    bad_values = corazon_df.loc[coerced.isna() & corazon_df[col].notna(), col]
    print(
        f"{col}: {bad_values.size} valores corruptos -> {sorted(bad_values.astype(str).unique())}"
    )

age: 2 valores corruptos -> ['fggfds', 'sdg']
rest_bp: 2 valores corruptos -> ['fsgh', 'wety']
chol: 2 valores corruptos -> ['sfdywe', 'wtey']
fbs: 0 valores corruptos -> []
max_hr: 1 valores corruptos -> ['adfs']
exang: 2 valores corruptos -> ['adfs', 'f']
old_peak: 2 valores corruptos -> ['afd', 'asd']
slope: 1 valores corruptos -> ['afd']
ca: 1 valores corruptos -> ['afd']
disease: 5 valores corruptos -> ['fsdg', 'fsg', 'g', 'gsfdg', 'sf']


In [8]:
# Valores de las columnas categóricas (se detectan números sin sentido y espacios extra)
categorical_columns = ["sex", "chest_pain", "rest_ecg", "thal"]
for col in categorical_columns:
    print(f"{col}: {sorted(corazon_df[col].dropna().astype(str).unique())}")

sex: ['2345', '45', '765', 'Female', 'Male']
chest_pain: ['2345', '2435', '3456', 'asymptomatic', 'nonanginal', 'nontypical', 'typical']
rest_ecg: ['3563', '36653', '435647', '5653', '5678', 'ST-T wave abnormality', 'left ventricular hypertrophy ', 'normal']
thal: ['365635463', '53646', '56', '87654', 'fixed', 'normal', 'reversable']


In [9]:
# Balance de la variable objetivo (solo valores válidos)
raw_target = corazon_df["disease"]
target = pd.to_numeric(raw_target, errors="coerce")
print(f"NaN originales en el target: {raw_target.isna().sum()}")
print(f"Valores corruptos en el target: {target.isna().sum() - raw_target.isna().sum()}")
print(f"Etiquetas validas: {target.isin([0, 1]).sum()} de {len(corazon_df)}")
target.value_counts().to_frame("pacientes")

NaN originales en el target: 106
Valores corruptos en el target: 5
Etiquetas validas: 2919 de 3030


,pacientes
disease,
0.0,1576
1.0,1343


### Síntesis del diagnóstico de calidad

| Problema | Magnitud | Ejemplos |
| --- | --- | --- |
| Filas duplicadas exactas | **2462 de 3030** (81.2%); solo 568 filas únicas | - |
| Valores nulos | 318 filas con ≥1 nulo; 15 filas completamente vacías | `rest_ecg` (193), `max_hr` (171), `ca` (162) |
| Texto basura en columnas numéricas | 18 celdas | `fggfds` en `age`, `wety` en `rest_bp`, `fsg` en `disease` |
| Números sin sentido en columnas categóricas | 15 celdas | `2345` en `sex`, `365635463` en `thal` |
| Espacios sobrantes en categorías | `rest_ecg` | `'left ventricular hypertrophy '` (espacio final) |
| Etiquetas contradictorias | 5 grupos de exámenes idénticos con `disease` distinto | - |
| Target inválido | 106 NaN + 5 valores corruptos | quedan 2919 etiquetas válidas |

**Balance de clases** (filas con etiqueta válida): 1576 sin enfermedad (54.0%) vs
1343 con enfermedad (46.0%) → dataset **aproximadamente balanceado**.

Estos problemas NO se corrigen en este notebook: aquí solo se documentan. La
limpieza se realizará en la siguiente iteración (notebook de preparación de datos).

## 🎯 Entendimiento del problema

Checklist del
[issue #7](https://github.com/ronaldo-duran/Hearth-project/issues/7):

**1. ¿Cuál es el objetivo del problema?**

Predecir si un paciente tiene una enfermedad cardíaca (`disease` = 1) o no
(`disease` = 0) a partir de 13 variables de sus exámenes médicos. Es un problema
de **clasificación binaria** con datos tabulares.

**2. ¿Cómo se usará su solución?**

Como **herramienta de apoyo al diagnóstico (screening)** para profesionales de
salud: ayudar a priorizar pacientes y complementar —no reemplazar— el criterio
clínico. En el contexto académico del curso, la solución es un modelo entrenado
offline que se evalúa con métricas de clasificación.

**3. ¿Cuáles son las soluciones actuales (si las hay)?**

- Diagnóstico clínico manual: un cardiólogo interpreta estos mismos exámenes
  (ECG en reposo, prueba de esfuerzo, fluoroscopia, talio).
- Scores clínicos de riesgo cardiovascular validados: Framingham, SCORE2, ASCVD.
- El estándar de referencia diagnóstico es la **coronariografía** (invasiva,
  costosa y con riesgos), que fue la base de la etiqueta del dataset original.

**4. ¿Cómo se debe enmarcar este problema?**

- Aprendizaje **supervisado** (cada fila tiene etiqueta `disease`).
- Tarea de **clasificación binaria**.
- Entrenamiento **offline/batch** (dataset estático, no hay flujo de datos en
  tiempo real); la predicción podría servirse en batch o de forma online.

**5. ¿Cómo se debe medir el desempeño de la solución (primera intuición)?**

Primera intuición: *accuracy*. Pero en un problema médico lo crítico es **no dejar
pasar enfermos** (falsos negativos), por lo que la métrica principal debería ser
el **recall (sensibilidad)**, complementado con **precision** y **F1-score**, y
reportando el **ROC-AUC** como medida global. Como el dataset está ~54/46
balanceado, la accuracy no es engañosa aquí, pero sola no basta.

**6. ¿La medida de desempeño está alineada con el objetivo del problema?**

Sí: recall/precision/F1 + ROC-AUC reflejan directamente el costo asimétrico de
los errores diagnósticos. La accuracy por sí sola NO estaría alineada, porque un
modelo trivial que prediga siempre "sin enfermedad" alcanzaría ~54% sin detectar
ningún enfermo.

**7. ¿Cuál sería el desempeño mínimo necesario para alcanzar el objetivo?**

- **Piso**: superar el baseline trivial (predecir siempre la clase mayoritaria ≈
  54% de accuracy / ROC-AUC 0.5).
- Referencia de la literatura sobre este dataset (Cleveland): modelos clásicos
  (regresión logística, random forest, XGBoost) alcanzan ~80-88% de accuracy y
  ROC-AUC ~0.9. Un mínimo razonable para el proyecto: **accuracy ≥ 80% con
  recall ≥ 80%** en el conjunto de prueba, tras la limpieza de datos.

**8. ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?**

Actualmente deben haber miles de repositorios publicos con esto, pero además desde la experiencia personal, trabajé con algo parecido en un proyecto de aula [ML-Dengue-Valle-Aburra](https://github.com/ronaldo-duran/ML-Dengue-Valle-Aburra)

**9. ¿Hay experiencia del problema disponible?**

Sí, abundante: es un dataset público muy estudiado (miles de notebooks públicos
en Kaggle/GitHub y publicaciones académicas), y el curso provee el template de
proyecto y la metodología de trabajo.

**10. (Importante) ¿Cómo se puede resolver el problema manualmente?**

Un cardiólogo revisa la edad y sexo del paciente, el tipo de dolor torácico, la
presión arterial en reposo, el colesterol, el ECG en reposo y los resultados de
la prueba de esfuerzo (frecuencia máxima, angina inducida, depresión ST,
pendiente ST), la fluoroscopia (`ca`) y la gammagrafía de talio (`thal`), y con
ese criterio clínico decide si hay enfermedad. 

**11. Listado de supuestos hasta este momento**

1. Cada fila representa un examen/paciente; las 3030 filas provienen de replicar
   y corromper el dataset original de ~303 pacientes (3030 ≈ 303 × 10).
2. La etiqueta `disease` corresponde a un diagnóstico confirmado (verdad de
   referencia).
3. Todas las variables predictoras se conocen **antes** del diagnóstico → no hay
   fuga de datos (data leakage) por columnas.
4. La versión modificada solo introduce ruido (duplicados, NaN, texto basura);
   no cambia el significado clínico de las variables.
5. Los problemas de calidad se pueden corregir (deduplicar, imputar) sin sesgar
   gravemente el modelo.
6. La muestra del hospital Cleveland (años 80) es representativa de la población
   objetivo — supuesto fuerte que habría que validar en un caso real.
7. Para separar train/test hay que **deduplicar primero**, si no el mismo
   examen aparecería en ambos conjuntos y el desempeño quedaría sobreestimado.

**12. ¿Cuál es la fuente de los datos?**

Kaggle [ronitf/heart-disease-uci](https://www.kaggle.com/ronitf/heart-disease-uci)
(versión original del hospital Cleveland del UCI ML Repository), en su **versión
modificada** por el curso. Copia local: `data/01_raw/corazon.csv` con su
diccionario en `data/01_raw/datos_corazon_Info.txt`. No se debe usar el dataset
original.

**13. ¿Cómo se actualizan los datos?**

No se actualizan: es un dataset estático de un estudio retrospectivo (1988). La
copia local en `data/01_raw` es inmutable (single source of truth según la
convención de capas de datos del proyecto).

**14. ¿Cada cuánto tiempo se actualizan los datos?**

No hay periodicidad (dataset estático). Si el sistema se llevara a producción,
habría que definir la captura prospectiva de nuevos exámenes y el
re-entrenamiento periódico del modelo.